# 02 · Exploratory Data Analysis

Digital Onboarding Optimization Case Study

> Fully synthetic dataset. No real company, customer, or proprietary information is included.

## 1. Executive Summary

This notebook explores the synthetic onboarding dataset to understand the overall shape of the user base, data quality, conversion profile, upload behavior, and early signals behind onboarding friction.

The main business hypothesis is:

> Users who postpone document submission are substantially less likely to complete identity verification and more likely to generate support demand.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

users = pd.read_csv(DATA_DIR / "onboarding_users.csv", parse_dates=["signup_timestamp"])
events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=["event_timestamp"])
support = pd.read_csv(DATA_DIR / "support_calls.csv", parse_dates=["call_timestamp"])

users.head()

## 2. Data Quality Assessment

In [ ]:
quality = pd.DataFrame({
    "dataset": ["users", "events", "support_calls"],
    "rows": [len(users), len(events), len(support)],
    "columns": [users.shape[1], events.shape[1], support.shape[1]],
    "duplicate_rows": [users.duplicated().sum(), events.duplicated().sum(), support.duplicated().sum()],
    "missing_cells": [users.isna().sum().sum(), events.isna().sum().sum(), support.isna().sum().sum()]
})
quality

In [ ]:
missing = users.isna().mean().sort_values(ascending=False).to_frame("missing_rate")
missing[missing["missing_rate"] > 0]

## 3. KPI Overview

In [ ]:
kpis = {
    "users": len(users),
    "approval_rate": users["approved"].mean(),
    "upload_now_rate": users["upload_choice"].eq("upload_now").mean(),
    "upload_later_rate": users["upload_choice"].eq("upload_later").mean(),
    "support_contact_rate": users["user_id"].isin(support["user_id"]).mean(),
    "avg_minutes_to_upload": users["minutes_to_upload"].mean()
}
pd.Series(kpis).to_frame("value")

## 4. User Distribution

In [ ]:
def plot_rate_by_category(df, category, metric, title, ylabel="Rate"):
    tmp = df.groupby(category)[metric].mean().sort_values(ascending=False)
    ax = tmp.plot(kind="bar", figsize=(10, 5))
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(category)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()
    return tmp.to_frame(metric)

for col in ["channel", "device", "region", "customer_type", "account_profile"]:
    display(users[col].value_counts(normalize=True).rename("share").to_frame())

## 5. Upload Behavior by Segment

In [ ]:
for col in ["channel", "device", "region", "customer_type", "account_profile"]:
    plot_rate_by_category(users.assign(upload_now=(users["upload_choice"] == "upload_now").astype(int)), col, "upload_now", f"Upload-now rate by {col}")

## 6. Approval Analysis

In [ ]:
approval_by_choice = users.groupby("upload_choice").agg(
    users=("user_id", "count"),
    approval_rate=("approved", "mean"),
    p1_completion_rate=("p1_completed", "mean"),
    p2_reach_rate=("p2_reached", "mean"),
    document_submission_rate=("document_submitted", "mean"),
    avg_minutes_to_upload=("minutes_to_upload", "mean")
)
approval_by_choice

In [ ]:
ax = approval_by_choice["approval_rate"].plot(kind="bar", figsize=(8, 5))
ax.set_title("Approval rate by document upload choice")
ax.set_ylabel("Approval rate")
ax.set_xlabel("Upload choice")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Time Analysis

In [ ]:
monthly = users.groupby("signup_month").agg(
    users=("user_id","count"),
    approval_rate=("approved","mean"),
    upload_now_rate=("upload_choice", lambda s: (s == "upload_now").mean()),
    support_rate=("user_id", lambda s: s.isin(support["user_id"]).mean())
)
monthly

In [ ]:
ax = monthly[["approval_rate", "upload_now_rate", "support_rate"]].plot(figsize=(11, 5), marker="o")
ax.set_title("Monthly KPI Trends")
ax.set_ylabel("Rate")
ax.set_xlabel("Signup month")
plt.tight_layout()
plt.show()

## 8. Correlation Matrix

In [ ]:
numeric_cols = [
    "has_document_ready", "is_business_hours", "p1_completed", "p2_reached",
    "document_submitted", "approved", "minutes_to_upload"
]
corr = users[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation Matrix")
fig.colorbar(im)
plt.tight_layout()
plt.show()

corr

## 9. Early Business Insights

1. Immediate document upload is strongly associated with higher approval.
2. Upload-later behavior is the main behavioral friction to investigate.
3. Acquisition channel differences suggest user intent and campaign quality matter.
4. Support contact rate can be treated as an operational cost proxy.
5. The next notebook should decompose the onboarding journey as a funnel.